# Notebook 2: The Stress Recovery Clock

An exploratory analysis of post-mainshock recovery dynamics for M7+ earthquakes, using b-value recovery time constants, Modified Omori Law fitting, and inter-event time analysis.

**Important methodological notes:** This notebook uses binned least-squares fitting (not maximum likelihood estimation) for Omori parameters, a simple spatial radius for aftershock selection (not a standard declustering algorithm such as Gardner-Knopoff or Reasenberg), and an assumed exponential recovery model that has not been tested against alternatives. These choices make this a first-pass, exploratory analysis. Results should be interpreted with these limitations in mind, and the paths for improvement are discussed in the summary.

In [1]:
import matplotlib
matplotlib.use('Agg')

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, ".")
from src.catalog import estimate_mc
from src.gutenberg_richter import estimate_b_value, rolling_b_value
from src.omori import fit_omori, compute_recovery_fraction, fit_recovery_exponential
from src.spatial import haversine, events_in_radius
from src.interevent import compute_interevent_times, rolling_cv
from src.plotting import setup_style, save_figure, plot_recovery_gallery
from src.external_data import load_pb2002_boundaries, classify_tectonic_setting

setup_style()

In [2]:
df = pd.read_csv("data/earthquake_catalog_global.csv")
df["time"] = pd.to_datetime(df["time"], format="ISO8601", utc=True)
print(f"Loaded {len(df):,} events")

Loaded 681,450 events


## 2.1 Mainshock Selection

M7+ events are selected from the global catalog and declustered by retaining only the largest event within 100 km and 30 days. This simple spatial/temporal grouping removes obvious doublets but is not a formal declustering algorithm.

In [3]:
# Select M7+ events
m7plus = df[df["mag"] >= 7.0].copy().sort_values("time").reset_index(drop=True)
print(f"M7+ events before declustering: {len(m7plus)}")

# Decluster: keep only the largest event within 100 km / 30 days
declustered = []
used = set()
for i, row in m7plus.iterrows():
    if i in used:
        continue
    # Find all M7+ within 100km and 30 days
    cluster = [i]
    for j, row2 in m7plus.iterrows():
        if j <= i or j in used:
            continue
        dist = haversine(row["latitude"], row["longitude"], row2["latitude"], row2["longitude"])
        dt = abs((pd.Timestamp(row2["time"]) - pd.Timestamp(row["time"])).total_seconds()) / 86400
        if dist <= 100 and dt <= 30:
            cluster.append(j)
    # Keep the largest
    best = max(cluster, key=lambda k: m7plus.loc[k, "mag"])
    declustered.append(m7plus.loc[best])
    used.update(cluster)

mainshocks = pd.DataFrame(declustered).reset_index(drop=True)
print(f"Mainshocks after declustering: {len(mainshocks)}")

M7+ events before declustering: 387


Mainshocks after declustering: 362


## 2.2 Recovery Analysis

For each mainshock, aftershocks are identified as all catalog events within a magnitude-dependent radius (R = 10^(0.5M - 1.8) km) over 1000 days. This is a simple spatial selection, not a standard declustering method (e.g., Gardner-Knopoff, 1974; Reasenberg, 1985), and it will include some independent background seismicity while potentially missing aftershocks outside the fixed radius.

The rolling b-value is computed in the aftershock zone, and a recovery fraction is derived relative to the pre-mainshock baseline. An exponential model f(t) = 1 - exp(-t/τ) is fit to the recovery fraction by least squares. This exponential form is assumed without testing against alternatives (power-law, logarithmic). The Modified Omori Law is fit to aftershock rates using binned least-squares via `scipy.optimize.curve_fit`, with parameter uncertainties derived from the covariance matrix. This differs from the standard MLE approach (Ogata, 1983) and is expected to produce biased parameter estimates (see discussion in Section 2.6).

In [4]:
recovery_results = []

for idx, ms in mainshocks.iterrows():
    ms_time = pd.Timestamp(ms["time"])
    if ms_time.tzinfo is None:
        ms_time = ms_time.tz_localize("UTC")
    ms_mag = ms["mag"]
    ms_lat = ms["latitude"]
    ms_lon = ms["longitude"]

    # Aftershock radius: R = 10^(0.5*M - 1.8) km
    radius_km = 10 ** (0.5 * ms_mag - 1.8)

    # Collect aftershocks (1000 days)
    t_end = ms_time + pd.Timedelta(days=1000)
    aftershocks = events_in_radius(df, ms_lat, ms_lon, radius_km)
    aftershocks = aftershocks[(aftershocks["time"] > ms_time) & (aftershocks["time"] <= t_end)]

    if len(aftershocks) < 30:
        continue

    mc = estimate_mc(aftershocks["mag"].values)

    # Pre-mainshock baseline (5 years before)
    t_pre_start = ms_time - pd.Timedelta(days=5*365)
    pre_events = events_in_radius(df, ms_lat, ms_lon, radius_km)
    pre_events = pre_events[(pre_events["time"] >= t_pre_start) & (pre_events["time"] < ms_time)]
    pre_above_mc = pre_events[pre_events["mag"] >= mc]

    if len(pre_above_mc) < 10:
        b_baseline = 1.0
    else:
        try:
            b_baseline, _ = estimate_b_value(pre_above_mc["mag"].values, mc)
        except ValueError:
            b_baseline = 1.0

    # Rolling b-value in aftershock zone
    as_above = aftershocks[aftershocks["mag"] >= mc]
    if len(as_above) < 60:
        continue

    rolling = rolling_b_value(as_above["time"].values, as_above["mag"].values,
                              mc=mc, window_size=min(50, len(as_above)//2), step=10)

    if len(rolling) < 3:
        continue

    # Recovery fraction — handle tz-naive center_time from rolling_b_value
    b_series = rolling["b_value"].values
    t_days = []
    for t in rolling["center_time"]:
        ct = pd.Timestamp(t)
        if ct.tzinfo is None:
            ct = ct.tz_localize("UTC")
        t_days.append((ct - ms_time).total_seconds() / 86400)
    t_days = np.array(t_days)

    recovery_frac = compute_recovery_fraction(b_series, b_baseline)
    fit_result = fit_recovery_exponential(recovery_frac, t_days)

    # Omori fit
    omori = fit_omori(aftershocks["time"], ms_time)

    result = {
        "name": ms.get("place", f"M{ms_mag:.1f}"),
        "time": ms_time,
        "mag": ms_mag,
        "lat": ms_lat,
        "lon": ms_lon,
        "n_aftershocks": len(aftershocks),
        "mc": mc,
        "b_baseline": b_baseline,
        "t_days": t_days,
        "b_values": b_series,
        "recovery_frac": recovery_frac,
        "tau_b": fit_result["tau_b"] if fit_result else np.nan,
        "recovery_r2": fit_result["r_squared"] if fit_result else np.nan,
        "omori_p": omori["p"] if omori else np.nan,
        "omori_c": omori["c"] if omori else np.nan,
        "omori_r2": omori["r_squared"] if omori else np.nan,
        "omori_p_std": omori["p_std"] if omori else np.nan,
        "omori_K_std": omori["K_std"] if omori else np.nan,
        "omori_c_std": omori["c_std"] if omori else np.nan,
    }
    recovery_results.append(result)

print(f"Successfully analyzed {len(recovery_results)} mainshock sequences")

Successfully analyzed 139 mainshock sequences


## 2.3 Recovery Curve Gallery

A quality filter (R² > 0.3) is applied to select well-constrained exponential recovery fits. As reported below, the pass rate is low, which suggests that the assumed exponential recovery model may not be appropriate for most aftershock sequences.

In [5]:
# Select 9 well-constrained examples (best R²)
all_results = recovery_results
well_constrained = [r for r in all_results if r["recovery_r2"] > 0.3]
well_constrained.sort(key=lambda r: r["mag"], reverse=True)
gallery_data = well_constrained[:9]

print(f"\nQuality filter: {len(well_constrained)} of {len(all_results)} sequences ({100*len(well_constrained)/len(all_results):.0f}%) pass R² > 0.3")
print("Note: The low pass rate suggests the exponential recovery model f(t) = 1 - exp(-t/\u03c4)")
print("may not be appropriate for most aftershock sequences. Alternative models (power-law,")
print("logarithmic) should be tested in future work.")

if len(gallery_data) > 0:
    fig = plot_recovery_gallery(gallery_data, n_rows=3, n_cols=3)
    save_figure(fig, "02_recovery_gallery")
    plt.show()
    print(f"Showing {len(gallery_data)} best-constrained recovery curves")
else:
    print("No well-constrained recovery curves found (R² > 0.3)")


Quality filter: 17 of 139 sequences (12%) pass R² > 0.3
Note: The low pass rate suggests the exponential recovery model f(t) = 1 - exp(-t/τ)
may not be appropriate for most aftershock sequences. Alternative models (power-law,
logarithmic) should be tested in future work.


Showing 9 best-constrained recovery curves


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5977/2383813599.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2.4 τ-b vs. Magnitude

Examining whether larger mainshocks produce longer b-value recovery times. Only sequences passing the R² > 0.3 quality filter are included. Omori parameter uncertainties (standard deviations from the curve_fit covariance matrix) are also reported below.

In [6]:
results_df = pd.DataFrame([{k: v for k, v in r.items()
                            if k not in ["t_days", "b_values", "recovery_frac"]}
                           for r in recovery_results])

well = results_df[results_df["recovery_r2"] > 0.3].copy()
print(f"Well-constrained recoveries: {len(well)} / {len(results_df)}")

if len(well) > 5:
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(well["mag"], well["tau_b"], c=well["recovery_r2"],
               cmap="viridis", s=50, alpha=0.7)
    plt.colorbar(ax.collections[0], ax=ax, label="R²")

    # Regression line
    from scipy import stats as sp_stats
    slope, intercept, r, p, se = sp_stats.linregress(well["mag"], well["tau_b"])
    x_fit = np.linspace(well["mag"].min(), well["mag"].max(), 50)
    ax.plot(x_fit, slope * x_fit + intercept, "r--", linewidth=1.5,
            label=f"Linear fit (R²={r**2:.2f}, p={p:.3f})")

    ax.set_xlabel("Mainshock Magnitude")
    ax.set_ylabel("τ_b (days)")
    ax.set_title("b-value Recovery Time vs. Mainshock Magnitude")
    ax.legend()
    save_figure(fig, "02_tau_b_vs_magnitude")
    plt.show()

# Report Omori parameter uncertainties
if "omori_p_std" in results_df.columns:
    p_stds = results_df["omori_p_std"].dropna()
    if len(p_stds) > 0:
        print(f"\nOmori parameter uncertainties (from curve_fit covariance):")
        print(f"  Median p uncertainty (std): {np.nanmedian(p_stds):.3f}")
        print(f"  Median K uncertainty (std): {np.nanmedian(results_df['omori_K_std'].dropna()):.3f}")
        print(f"  Median c uncertainty (std): {np.nanmedian(results_df['omori_c_std'].dropna()):.3f}")

Well-constrained recoveries: 17 / 139



Omori parameter uncertainties (from curve_fit covariance):
  Median p uncertainty (std): 0.490
  Median K uncertainty (std): 3012.032
  Median c uncertainty (std): 8.568


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5977/2324727298.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2.5 Global Map of Recovery Times

Spatial distribution of b-value recovery time constants for the well-constrained subset. Given that only 12% of sequences pass the quality filter, this map is necessarily sparse and should be viewed as illustrative rather than comprehensive.

In [7]:
if len(well) > 0:
    fig, ax = plt.subplots(figsize=(14, 7))
    sc = ax.scatter(well["lon"], well["lat"], c=well["tau_b"], cmap="plasma",
                    s=well["mag"]**2, alpha=0.7, edgecolors="k", linewidth=0.3)
    plt.colorbar(sc, ax=ax, label="τ_b (days)")
    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)
    ax.set_xlabel("Longitude (°)")
    ax.set_ylabel("Latitude (°)")
    ax.set_title("M7+ Mainshocks Colored by b-value Recovery Time")
    ax.set_aspect("equal")
    save_figure(fig, "02_global_recovery_map")
    plt.show()

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5977/2616920228.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2.6 Omori p-value Distribution

The distribution of Modified Omori Law p-values across all analyzed sequences. Note that these are derived from binned least-squares fitting, not MLE, so the absolute values should be interpreted cautiously.

In [8]:
valid_p = results_df["omori_p"].dropna()
if len(valid_p) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(valid_p, bins=30, color="#457B9D", alpha=0.7, edgecolor="white")
    ax.axvline(1.0, color="#E63946", linewidth=2, linestyle="--", label="p = 1.0")
    ax.set_xlabel("Omori p-value")
    ax.set_ylabel("Count")
    ax.set_title("Distribution of Modified Omori Law p-values")
    ax.legend()
    save_figure(fig, "02_omori_p_histogram")
    plt.show()
    print(f"Median p = {valid_p.median():.2f}, Mean p = {valid_p.mean():.2f}")

    # Report parameter uncertainties
    p_stds = results_df["omori_p_std"].dropna()
    if len(p_stds) > 0:
        print(f"Median p uncertainty (std): {np.nanmedian(p_stds):.3f}")

Median p = 1.86, Mean p = 1.98
Median p uncertainty (std): 0.490


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5977/2628100028.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Note on elevated p-values:** The median p of approximately 1.86 is substantially higher than the global consensus of p = 1.0-1.2 (Utsu et al., 1995). This inflation likely reflects methodological bias in the binned least-squares fitting approach: aftershock rates are estimated in 30-day windows with 10-day stride, producing highly smoothed, autocorrelated rate estimates that can steepen the apparent decay. The overlapping windows also violate the independence assumption of `curve_fit`, inflating apparent goodness-of-fit and distorting parameter estimates. Maximum likelihood estimation on individual event times (Ogata, 1983) would produce more accurate, directly comparable p-values. These elevated values should not be interpreted as physically meaningful departures from the literature consensus.

## 2.7 Recovery Dynamics by Tectonic Setting (PB2002)

Using the PB2002 plate boundary model, we classify each mainshock by its tectonic setting and test whether Omori p-values and aftershock productivity vary systematically across settings. A Kruskal-Wallis test is used to assess whether median p-values differ significantly between tectonic environments. As reported below, the test result is non-significant, meaning we cannot distinguish recovery dynamics by tectonic setting with this dataset and methodology.

In [9]:
# Classify each mainshock by tectonic setting
boundaries = load_pb2002_boundaries()
results_df["tectonic_setting"] = [
    classify_tectonic_setting(row["lat"], row["lon"], boundaries)
    for _, row in results_df.iterrows()
]

setting_counts = results_df["tectonic_setting"].value_counts()
print(f"Mainshocks by tectonic setting:\n{setting_counts}\n")

setting_order = ["subduction", "convergent", "transform", "spreading", "rift", "intraplate"]
setting_colors = {
    "subduction": "#E63946", "convergent": "#F4A261", "transform": "#457B9D",
    "spreading": "#2A9D8F", "rift": "#A8DADC", "intraplate": "#AAAAAA"
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# Panel 1: Omori p by tectonic setting
ax = axes[0]
plot_data = []
plot_labels = []
box_colors = []
for s in setting_order:
    vals = results_df[results_df["tectonic_setting"] == s]["omori_p"].dropna()
    if len(vals) >= 3:
        plot_data.append(vals.values)
        plot_labels.append(f"{s}\n(n={len(vals)})")
        box_colors.append(setting_colors[s])

if plot_data:
    bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True,
                    medianprops=dict(color="black", linewidth=2))
    for patch, color in zip(bp["boxes"], box_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8, label="p = 1")
ax.set_ylabel("Omori p-value")
ax.set_title("Aftershock Decay Rate by Setting")
ax.tick_params(axis="x", rotation=30)

# Panel 2: Number of aftershocks by setting
ax = axes[1]
plot_data = []
plot_labels = []
box_colors = []
for s in setting_order:
    vals = results_df[results_df["tectonic_setting"] == s]["n_aftershocks"]
    if len(vals) >= 3:
        plot_data.append(vals.values)
        plot_labels.append(f"{s}\n(n={len(vals)})")
        box_colors.append(setting_colors[s])

if plot_data:
    bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True,
                    medianprops=dict(color="black", linewidth=2))
    for patch, color in zip(bp["boxes"], box_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
ax.set_ylabel("Aftershock count (1000 days)")
ax.set_title("Aftershock Productivity by Setting")
ax.set_yscale("log")
ax.tick_params(axis="x", rotation=30)

# Panel 3: Mainshock map colored by setting
ax = axes[2]
for s in setting_order:
    mask = results_df["tectonic_setting"] == s
    if mask.sum() > 0:
        ax.scatter(results_df.loc[mask, "lon"], results_df.loc[mask, "lat"],
                   c=setting_colors[s], s=results_df.loc[mask, "mag"]**2 * 0.5,
                   alpha=0.6, label=s, edgecolors="k", linewidth=0.2)
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("M7+ Mainshocks by Tectonic Setting")
ax.legend(fontsize=7, loc="lower left", ncol=2)

fig.suptitle("Post-Mainshock Recovery by Tectonic Setting (PB2002)",
             fontsize=13, fontweight="bold")
fig.tight_layout()
save_figure(fig, "02_recovery_by_tectonic_setting")
plt.show()

# Summary statistics
from scipy.stats import kruskal
print("\nOmori p by tectonic setting:")
for s in setting_order:
    vals = results_df[results_df["tectonic_setting"] == s]["omori_p"].dropna()
    if len(vals) >= 3:
        print(f"  {s:15s}: median p={vals.median():.2f}, mean={vals.mean():.2f}, n={len(vals)}")

# Kruskal-Wallis on Omori p
groups = [results_df[results_df["tectonic_setting"] == s]["omori_p"].dropna().values
          for s in setting_order
          if len(results_df[results_df["tectonic_setting"] == s]["omori_p"].dropna()) >= 3]
if len(groups) >= 2:
    H, kw_p = kruskal(*groups)
    print(f"\nKruskal-Wallis on Omori p: H = {H:.1f}, p = {kw_p:.2e}")
    if kw_p > 0.05:
        print(f"Result: No statistically significant difference in Omori p across tectonic settings (p = {kw_p:.3f} > 0.05).")
    else:
        print(f"Result: Statistically significant difference in Omori p across tectonic settings (p = {kw_p:.3f}).")

print("\nAftershock productivity by tectonic setting:")
for s in setting_order:
    vals = results_df[results_df["tectonic_setting"] == s]["n_aftershocks"]
    if len(vals) >= 3:
        print(f"  {s:15s}: median={vals.median():.0f}, mean={vals.mean():.0f}, n={len(vals)}")

Mainshocks by tectonic setting:
tectonic_setting
subduction    100
convergent     13
transform      12
intraplate      8
spreading       4
rift            2
Name: count, dtype: int64



/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5977/2732917288.py:32: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True,
/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5977/2732917288.py:55: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True,



Omori p by tectonic setting:
  subduction     : median p=1.77, mean=1.97, n=100
  convergent     : median p=2.36, mean=2.28, n=12
  transform      : median p=2.27, mean=2.30, n=12
  spreading      : median p=0.42, mean=0.87, n=4
  intraplate     : median p=1.71, mean=1.54, n=8

Kruskal-Wallis on Omori p: H = 6.3, p = 1.78e-01
Result: No statistically significant difference in Omori p across tectonic settings (p = 0.178 > 0.05).

Aftershock productivity by tectonic setting:
  subduction     : median=372, mean=770, n=100
  convergent     : median=344, mean=525, n=13
  transform      : median=265, mean=606, n=12
  spreading      : median=302, mean=544, n=4
  intraplate     : median=486, mean=699, n=8


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5977/2732917288.py:84: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

This notebook presents an exploratory, first-pass analysis of post-mainshock recovery dynamics for M7+ earthquakes worldwide.

### Key results

- **Omori p-values:** The median fitted p is approximately 1.86, substantially above the literature consensus of p = 1.0-1.2. This inflation is attributed to the binned least-squares fitting method (30-day windows, 10-day stride), which introduces smoothing and autocorrelation bias. These values should not be interpreted as accurate physical measurements.
- **b-value recovery:** Only 17 of 139 analyzed sequences (12%) yield well-constrained exponential recovery fits (R² > 0.3). The low pass rate indicates the assumed exponential model f(t) = 1 - exp(-t/τ) is not appropriate for the majority of aftershock sequences.
- **Tectonic setting dependence:** A Kruskal-Wallis test on Omori p-values across tectonic settings is non-significant (p = 0.178), meaning we find no evidence that recovery dynamics differ by tectonic environment in this analysis.

### Limitations

This analysis has several important methodological limitations:

1. **Binned least-squares vs. MLE:** Omori parameters are fit using binned rates and `scipy.optimize.curve_fit`, not the standard maximum likelihood estimation on individual event times (Ogata, 1983). This is known to produce biased parameter estimates, particularly inflated p-values.
2. **No standard declustering:** Aftershocks are identified using a simple magnitude-dependent spatial radius, not an established declustering algorithm (e.g., Gardner-Knopoff, 1974; Reasenberg, 1985). This will include some background seismicity and may miss aftershocks outside the fixed radius.
3. **Exponential recovery model untested:** The b-value recovery is fit to an exponential model without testing against plausible alternatives (power-law, logarithmic). The low 12% pass rate suggests this functional form is frequently inappropriate.
4. **Parameter uncertainties from curve_fit covariance:** While uncertainties are reported, they are derived from the Hessian approximation in least-squares fitting, which underestimates true uncertainty when model assumptions (independence, homoscedasticity) are violated.

### Paths for improvement

- Implement MLE-based Omori fitting (Ogata, 1983) for unbiased p-value estimates
- Apply standard declustering (Gardner-Knopoff or Reasenberg) for more rigorous aftershock identification
- Test multiple recovery functional forms (exponential, power-law, logarithmic) and use model selection criteria (AIC/BIC)
- Increase the quality filter pass rate by exploring alternative recovery metrics beyond b-value